# Open Notebook in Colab


[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/childmindresearch/llm_tracker/blob/main/tutorials/anonymization_tutorial.ipynb)


# Removing personal information before LLM coding

**TL;DR** — `llm_tracker` sends your documents to a language model over the internet, so
anything in them that identifies a person leaves your computer. This notebook strips
names, phone numbers, addresses and similar details **on your own machine first**, using
[`anonymize-pii`](https://github.com/childmindresearch/anonymize-pii). Fill in the
settings box in **section 0**, run the cells in order, and do not skip **section 5**:
no automatic tool catches everything.

**What the notebook does, in order**

1. Installs the tool (once per computer).
2. Reads your documents — a spreadsheet, or a folder of `.txt` / `.docx` files
   (interview transcripts, reports).
3. Finds personal details and replaces each one with a label, e.g.
   `Ana López called on 388-155-4321` → `<PERSON> called on <PHONE_NUMBER>`.
4. Helps you **check the result** and fix what it got wrong.
5. Hands the cleaned text to `llm_tracker`.

**What you need**

- Python **3.12 or newer** and `git` (a program that downloads code from the internet).
- A few GB of free disk space: the tool downloads three detection models.
- Time: on a normal laptop, expect seconds to a couple of minutes **per document**.
  A graphics card (GPU) speeds this up but is not required.

> ⚠️ **No automatic tool removes 100% of personal information.** Treat the result as a
> first pass that a human still has to read. Keep the original, un-anonymised files out
> of any folder that is synced to the cloud, shared, or under version control.


## Contents

| section | what it is for | do I have to run it? |
| --- | --- | --- |
| [0. Settings](#0-settings) | the one cell you must edit: paths, your files, language | **required** |
| [1. Install](#1-install) | download the tool and its models | **required**, once per computer |
| [2. Language](#2-language) | switch the detectors from English to your language | required **only if your text is not in English** |
| [3. Load your documents](#3-load-your-documents) | spreadsheet, folder of `.txt`/`.docx`, or the built-in example | **required** |
| [4. Run the anonymiser](#4-run-the-anonymiser) | the actual anonymisation | **required** |
| [5. Check the results](#5-check-the-results) | read what it removed, and what it missed | **required — this is the important part** |
| [6. Stop it deleting clinical words](#6-stop-it-deleting-clinical-words) | protect vocabulary like `Bipolar` or `Ritalin` | optional, but usual after a first look |
| [7. Catch names the detectors missed](#7-catch-names-the-detectors-missed) | extra rules for forms and ALL-CAPS text | optional — recommended for clinical records |
| [8. Send the clean text to llm_tracker](#8-send-the-clean-text-to-llm_tracker) | continue with the coding pipeline | required only if that is your goal |
| [9. Checklist before anything leaves your computer](#9-checklist-before-anything-leaves-your-computer) | final review | **required** |
| [Appendix A: how the pipeline works](#appendix-a-how-the-pipeline-works) | background, useful when a result surprises you | reading only |
| [Appendix B: anonymising only part of a document](#appendix-b-anonymising-only-part-of-a-document) | long reports with headings | advanced, optional |

A section marked *optional* can be skipped entirely on a first pass — nothing later
breaks. The ones marked **required** have to run in order.


## 0. Settings

**Everything you need to change is in the next cell.** The rest of the notebook reads
these values, so you should not have to edit code further down. Run this cell first,
and run it again whenever you change one of the values.

Quick guide to the ones that matter most:

| setting | what to put in it |
| --- | --- |
| `INPUT_KIND` | `"sample"` to try the notebook with fake data, `"csv"` for a spreadsheet, `"folder"` for a folder of `.txt` / `.docx` files (e.g. interview transcripts). |
| `CSV_TEXT_COLUMN` | the column that holds the text to be cleaned. **Only free text**, never columns with fixed options like sex or diagnosis code (see the warning in section 3). |
| `CSV_ID_COLUMN` | the column that identifies each row (a case number). These identifiers stay attached to the text, so you can trace a result back to the original record. |
| `LANGUAGE` | `"en"`, `"es"`, `"pt"`… the language your **documents** are written in. |
| `MUST_NOT_APPEAR` | a handful of real names / phone numbers you know are in your own files. Section 5.3 searches the output for them. Fill this in — it is the single most useful check in the notebook. |
| `SKIPLIST_TERMS` | words that must never be removed, even if a detector thinks they are a name (e.g. `Bipolar`, `Ritalin`). You will add to this after your first run. |

One more editable list lives further down because it is long: the guard words in
**section 7.2**, used only if you run the extra name rules.


In [ ]:
from pathlib import Path

# --- where things live -----------------------------------------------------
PROJECT_DIR = Path.cwd()                    # the folder this notebook is in
ANON_DIR = PROJECT_DIR / "anonymize-pii"    # the tool; section 1 downloads it here

# --- your documents --------------------------------------------------------
INPUT_KIND = "sample"          # "sample" | "csv" | "folder"

INPUT_CSV = ""                 # INPUT_KIND = "csv": path to the .csv file
CSV_TEXT_COLUMN = "text"       #   the column with the text to anonymise
CSV_ID_COLUMN = "doc_id"       #   the column identifying each row

INPUT_FOLDER = ""              # INPUT_KIND = "folder": folder of .txt / .docx files

# --- language of your documents --------------------------------------------
LANGUAGE = "en"                # "en", "es", "pt", "fr", "de", "it", ...

# --- how the removal looks -------------------------------------------------
MASK_MODE = "entity"           # "entity" -> <PERSON>, <PHONE_NUMBER>
                               # "redact" -> <REDACTED> for everything

# --- checking (section 5 and 6) --------------------------------------------
# Real identifiers you KNOW appear in your own documents. Section 5.3 checks that
# none of them survived. Replace these examples with your own.
MUST_NOT_APPEAR = [
    "Aisha Patel",
    "323-555-0891",
    "281 Pleasant Boulevard",
]

# Words that must never be removed, even if a detector flags them.
SKIPLIST_TERMS = [
    "Bipolar",
    "Ritalin",
]

# --- llm_tracker, section 8 (leave empty if you stop after anonymising) -----
OPENROUTER_API_KEY = ""                            # key, or path to a .env file
LLM_MODEL = "google/gemini-3-flash-preview"        # any model from openrouter.ai/models
CODEBOOK_PATH = "codebook.json"

# --- derived paths: no need to change --------------------------------------
SRC_DIR = ANON_DIR / "src" / "anonymize_pii"   # the tool must be run from here
RAW_DIR = ANON_DIR / "data" / "raw"            # where the input is written
EXPORT_DIR = ANON_DIR / "data" / "exports"     # where the results appear

# The Python that has the tool's dependencies. With `uv sync` (option A in section 1)
# it is the tool's own environment; with pip (option B) it is this notebook's Python.
PYTHON = str(ANON_DIR / ".venv" / "bin" / "python")    # Windows: ".venv/Scripts/python.exe"
# import sys; PYTHON = sys.executable                  # uncomment for option B / Colab

assert INPUT_KIND in {"sample", "csv", "folder"}, "INPUT_KIND must be sample, csv or folder"
assert MASK_MODE in {"entity", "redact"}, "MASK_MODE must be entity or redact"
print(f"tool folder : {ANON_DIR}")
print(f"input       : {INPUT_KIND}")
print(f"language    : {LANGUAGE}")


## 1. Install

**TL;DR** — run the three cells below once per computer.
Everything here happens on your machine. Nothing is uploaded.


In [ ]:
!git clone https://github.com/childmindresearch/anonymize-pii.git "{ANON_DIR}"


### Option A — `uv` (recommended)

`uv` is an installer that reads the exact list of versions shipped with the tool, so you
get the same setup the authors tested, detection models included. Run **either** option A
**or** option B, not both.


In [ ]:
# Install uv itself if you don't have it (skip this line otherwise):
!curl -LsSf https://astral.sh/uv/install.sh | sh

!cd "{ANON_DIR}" && uv sync


### Option B — `pip`

`pip` is the standard Python installer; use it if `uv` gives you trouble.
`requirements.txt` pins the same versions, but it leaves out `headhunter`, which is only
needed for the optional step in Appendix B. If you take this route, remember to set
`PYTHON = sys.executable` in section 0 (the commented line).


In [ ]:
!pip install -r "{ANON_DIR}/requirements.txt"

# Only needed for Appendix B:
!pip install "git+https://github.com/childmindresearch/headhunter"


Two more downloads happen automatically the **first time you run the anonymiser**, and
are then cached on disk:

- the Stanza language models → `~/stanza_resources`
- the GLiNER model `nvidia/gliner-pii` → your Hugging Face cache

If the first run seems to hang for several minutes with no output, this is usually why.


## 2. Language

> **Skip this section if your documents are in English.**

**TL;DR** — out of the box the three detectors only understand English; on Spanish text
they find almost nothing. The cell below rewrites the tool's two configuration files so
they work in the language you set as `LANGUAGE` in section 0. Run it once, after
installing and before section 4.

First download the language models for that language (`es` shown; replace with yours):


In [ ]:
# Spanish: es_core_news_lg / es_core_news_sm. Same shape for pt, fr, de, it.
big_model = f"{LANGUAGE}_core_news_lg"
small_model = f"{LANGUAGE}_core_news_sm"

!cd "{ANON_DIR}" && uv run python -m spacy download {big_model}
!cd "{ANON_DIR}" && uv run python -m spacy download {small_model}

# With pip (option B), drop the `uv run` part:
# !python -m spacy download es_core_news_lg


Now switch the pipeline over. The cell keeps a copy of each original file next to it
(`config.py.orig`, `anonymizers.py.orig`), so it is safe to re-run and easy to undo.


In [ ]:
import re
import shutil

SPACY_MODELS = {                      # the spaCy model names per language
    "en": ("en_core_web_lg", "en_core_web_sm"),
    "es": ("es_core_news_lg", "es_core_news_sm"),
    "pt": ("pt_core_news_lg", "pt_core_news_sm"),
    "fr": ("fr_core_news_lg", "fr_core_news_sm"),
    "de": ("de_core_news_lg", "de_core_news_sm"),
    "it": ("it_core_news_lg", "it_core_news_sm"),
}


def set_pipeline_language(src_dir, language):
    """Point the three detectors at `language`, working from a pristine backup."""
    src_dir = Path(src_dir)
    big, small = SPACY_MODELS.get(
        language, (f"{language}_core_news_lg", f"{language}_core_news_sm")
    )
    edits = {
        "config.py": [
            # the spaCy models, and the small one GLiNER borrows
            (r'\{"lang_code": "\w+", "model_name": "\w+_core_(?:web|news)_lg"\}',
             f'{{"lang_code": "{language}", "model_name": "{big}"}}'),
            (r'\{"lang_code": "\w+", "model_name": "\w+_core_(?:web|news)_sm"\}',
             f'{{"lang_code": "{language}", "model_name": "{small}"}}'),
            # the Stanza model
            (r'("nlp_engine_name": "stanza",\s*\n\s*"models": \[)'
             r'\{"lang_code": "\w+", "model_name": "\w+"\}\]',
             f'\\1{{"lang_code": "{language}", "model_name": "{language}"}}]'),
            # the rule-based recognizers (emails, phone numbers, ...)
            (r'registry\.load_predefined_recognizers\([^)]*\)',
             f'registry.load_predefined_recognizers(languages=["{language}"])'),
            # tell GLiNER which language it is reading
            (r'device=device,?(\s*\n\s*supported_language="\w+",?)?(\s*\n\s*)\)',
             f'device=device,\\2    supported_language="{language}"\\2)'),
        ],
        # only the scanning pass changes language; see the note below
        "anonymizers.py": [
            (r'(text=chunk,[ \t]*\n[ \t]*language=)"\w+"', f'\\1"{language}"'),
        ],
    }

    for filename, rules in edits.items():
        path = src_dir / filename
        backup = path.with_suffix(path.suffix + ".orig")
        if not backup.exists():
            shutil.copy(path, backup)          # keep the English original
        text = backup.read_text(encoding="utf-8")
        for pattern, replacement in rules:
            text, n_changes = re.subn(pattern, replacement, text)
            if n_changes == 0:
                raise RuntimeError(
                    f"{filename}: could not find the line to change "
                    f"({pattern}). The tool may have been updated - "
                    "edit it by hand, see the list at the end of this section."
                )
        path.write_text(text, encoding="utf-8")
        print(f"{filename}: now reading {language!r}")


if LANGUAGE != "en":
    set_pipeline_language(SRC_DIR, LANGUAGE)
else:
    print("LANGUAGE is 'en' - nothing to change.")


### What to expect in another language

The pipeline works, but not equally well. From running it over Spanish-language clinical
records:

- **ID numbers and local phone formats are not recognised.** The built-in rules for
  identity documents are US-specific (social security, driver's licence, passport). For a
  DNI, CUIT, NIF or a local phone format, add your own search-and-replace pass — do not
  assume the detectors handle them.
- **ALL-CAPS text breaks name detection**, and administrative records are full of it.
  Section 7 measures this and adds rules that recover what is lost.
- **Do not convert your text to Title Case to help the detectors.** It does find a few
  more names, but it invents many more false ones (also measured in section 7).
- **Only run free text through it.** On columns with fixed options the detectors produce
  confident nonsense — a sex value flagged as `<PERSON>`, a diagnosis label as
  `<ORGANIZATION>` — which destroys the column and buys no privacy.

<details>
<summary><b>The same changes, by hand (if the cell above fails)</b></summary>

In `src/anonymize_pii/config.py`:

```python
spacy = {'name': 'spacy', 'config': {
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "es", "model_name": "es_core_news_lg"}]}}

stanza = {'name': 'stanza', 'config': {
    "nlp_engine_name": "stanza",
    "models": [{"lang_code": "es", "model_name": "es"}]}}

GLiNER = {'name': 'GLiNER',
    'config': {"nlp_engine_name": "spacy",
               "models": [{"lang_code": "es", "model_name": "es_core_news_sm"}]},
    'external_model': "nvidia/gliner-pii"}
```

and, inside `get_warm_engines()`:

```python
registry.load_predefined_recognizers(languages=["es"])          # instead of ()

gliner_rec = GlinerRecognizer(
    model_name=config.get('external_model'),
    labels=Entities,
    device=device,
    supported_language="es",                                     # added
)
```

In `src/anonymize_pii/anonymizers.py`, inside `EntityScanner.scan()`:

```python
results = self.analyzer.analyze(text=chunk, language="es", entities=self.entities)
```

**Leave the other two `language="en"` in that file alone.** They belong to the
replacement step, which is a plain word-for-word lookup rather than detection; changing
them breaks the replacement without improving anything.

</details>


## 3. Load your documents

**TL;DR** — the anonymiser reads one file, `data/raw/Reports.json`, which pairs an
identifier with the full text of each document:

```json
{
  "entrevista_01": "Entrevistadora: Buenas tardes ...",
  "entrevista_02": "Informe de guardia ..."
}
```

The cell below builds that file from whichever `INPUT_KIND` you chose in section 0.
Those identifiers follow the text through every later file, so use the same ones you use
elsewhere — that is what lets you match a result back to the original record.

> ⚠️ **Put only free text through the anonymiser.** Columns with a fixed set of values —
> sex, diagnosis codes, town names, rating scales — should be de-identified with your own
> explicit rules (drop the column, group values, round coordinates). Running detectors
> over them corrupts the values and buys no privacy.

The three options:

- **`"sample"`** — a set of invented reports (fake names, fake addresses) that ships with
  the tool. Use it to see the notebook work end to end before touching real data.
- **`"csv"`** — a spreadsheet exported as CSV, one document per row.
- **`"folder"`** — a folder of `.txt` or `.docx` files, one document per file: interview
  transcripts, session notes, reports. The file name (without the extension) becomes the
  identifier, so `entrevista_01.docx` → `entrevista_01`. Subfolders are included.
  Old-style `.doc` files are not supported — open and re-save them as `.docx` first.


In [ ]:
import html
import json
import re
import shutil
import zipfile

import pandas as pd


def read_docx(path):
    """The plain text of a .docx: one line per paragraph and per table row."""
    try:
        import docx                      # python-docx, if it happens to be installed
    except ImportError:                  # otherwise read the file's XML directly
        with zipfile.ZipFile(path) as archive:
            xml = archive.read("word/document.xml").decode("utf-8", "replace")
        xml = re.sub(r"<w:(?:p|tr)\b[^>]*/?>", "\n", xml)     # paragraphs and table rows
        xml = re.sub(r"<w:tab\b[^>]*/?>", "\t", xml)
        return html.unescape(re.sub(r"<[^>]+>", "", xml)).strip()

    document = docx.Document(str(path))
    lines = [paragraph.text for paragraph in document.paragraphs]
    lines += ["\t".join(cell.text for cell in row.cells)
              for table in document.tables for row in table.rows]
    return "\n".join(lines).strip()


def load_folder(folder):
    """Every .txt / .md / .docx file in `folder` (and its subfolders) as {id: text}."""
    folder = Path(folder)
    if not folder.is_dir():
        raise ValueError(f"INPUT_FOLDER is not a folder: {folder}")

    documents = {}
    for path in sorted(folder.rglob("*")):
        suffix = path.suffix.lower()
        if suffix in {".txt", ".md"}:
            text = path.read_text(encoding="utf-8", errors="replace")
        elif suffix == ".docx":
            text = read_docx(path)
        else:
            continue                                   # anything else is ignored
        if not text.strip():
            print(f"  warning: {path.name} is empty, skipped")
            continue
        if path.stem in documents:
            raise ValueError(f"Two files are called {path.stem!r}; identifiers must be unique.")
        documents[path.stem] = text

    if not documents:
        raise ValueError(f"No .txt, .md or .docx files found in {folder}")
    return documents


def load_csv(path, id_column, text_column):
    """One row per document, as {id: text}."""
    table = pd.read_csv(path)
    for column in (id_column, text_column):
        if column not in table.columns:
            raise ValueError(f"Column {column!r} is not in {path}. Found: {list(table.columns)}")
    return dict(zip(table[id_column].astype(str), table[text_column].astype(str)))


RAW_DIR.mkdir(parents=True, exist_ok=True)

if INPUT_KIND == "sample":
    shutil.copy(ANON_DIR / "tests" / "Reports.json", RAW_DIR / "Reports.json")
    reports = json.loads((RAW_DIR / "Reports.json").read_text(encoding="utf-8"))
else:
    reports = (load_csv(INPUT_CSV, CSV_ID_COLUMN, CSV_TEXT_COLUMN)
               if INPUT_KIND == "csv" else load_folder(INPUT_FOLDER))
    with open(RAW_DIR / "Reports.json", "w", encoding="utf-8") as handle:
        json.dump(reports, handle, ensure_ascii=False, indent=2)

print(f"{len(reports)} documents -> {RAW_DIR / 'Reports.json'}")
print(f"identifiers: {list(reports)[:5]}{' ...' if len(reports) > 5 else ''}")

first_id = next(iter(reports))
print(f"\n--- start of {first_id} ---\n{reports[first_id][:400]}")


## 4. Run the anonymiser

**TL;DR** — this is the step that does the work. It reads `data/raw/Reports.json`, finds
the personal details, and writes the cleaned text to `data/exports`. Expect it to take a
while: it reads every document with three different detectors.

The cell runs the tool for you. If you would rather use a terminal:

```bash
cd anonymize-pii/src/anonymize_pii
uv run python main.py --mask entity --output merged
```

That `cd` is not optional. The tool works out where your data lives from the folder you
launch it in, and its files import each other by bare name, so from anywhere else it
either crashes or writes to the wrong place. The cell below passes the folder explicitly
for the same reason.


In [ ]:
import subprocess

result = subprocess.run(
    [PYTHON, "main.py", "--mask", MASK_MODE, "--output", "merged"],
    cwd=SRC_DIR,                 # required: the tool derives every path from this
    capture_output=True,
    text=True,
)

print(result.stdout[-3000:])
if result.returncode != 0:
    print("--- it failed; the error is below ---")
    print(result.stderr[-3000:])


### The options, if you want to change them

| flag | set by | what it does |
| --- | --- | --- |
| `--mask entity` | `MASK_MODE = "entity"` | each detail becomes its type: `<PERSON>`, `<LOCATION>`, `<PHONE_NUMBER>`… |
| `--mask redact` | `MASK_MODE = "redact"` | everything becomes the same `<REDACTED>` label |
| `--output merged` | fixed here | three result files covering all documents |
| `--output single` | — | one folder per document instead; useful for very large runs |
| `--parse` | — | pre-processes long reports so you can anonymise only certain sections (Appendix B) |

Keep `entity` while you are still reviewing: knowing *what kind* of detail was removed
makes the output far easier to check, and a language model reads `<PERSON> reported ...`
more naturally than `<REDACTED> reported ...`. Switch to `redact` if the labels
themselves would give too much away.


## 5. Check the results

> **This section is the reason the notebook exists. Do not skip it.**

**TL;DR** — the tool replaces every occurrence of a detail it *found*. Anything it did
**not** find stays in your text word for word. So the job here is to look for what is
missing, not to admire what was removed.

The run wrote three files into `data/exports`:

| file | what is in it |
| --- | --- |
| `Anonymized_Reports.json` | the cleaned text, in the same shape as your input |
| `Iterator.json` | every detail each detector flagged, with its type and how confident it was |
| `PII_Log.json` | one line per replacement: type, position in the text, score, which detector fired |


In [ ]:
with open(EXPORT_DIR / "Anonymized_Reports.json", encoding="utf-8") as handle:
    anonymized = json.load(handle)
with open(EXPORT_DIR / "Iterator.json", encoding="utf-8") as handle:
    iterator = json.load(handle)
with open(EXPORT_DIR / "PII_Log.json", encoding="utf-8") as handle:
    pii_log = json.load(handle)

first_id = next(iter(anonymized))
print(f"=== {first_id} ===")
print(anonymized[first_id][:1500])


### 5.1 What was flagged, and how sure was it

Each detection carries a score between 0 and 1. **Read the lowest ones first**: those are
the likeliest mistakes, i.e. real content about to be deleted.


In [ ]:
ENGINES = ["spacy", "stanza", "GLiNER"]

rows = [
    {
        "doc_id": doc_id,
        "engine": engine,
        "text": text,
        "entity_type": entity_type,
        "score": score,
    }
    for doc_id, per_engine in iterator.items()
    for engine in ENGINES
    for text, (entity_type, score) in per_engine.get(engine, {}).items()
]

flagged = pd.DataFrame(rows).sort_values("score")
print(f"{len(flagged)} detections, {flagged['text'].nunique()} distinct strings")
flagged.head(30)


### 5.2 Where the three detectors disagreed

A detail only needs **one** of the three detectors to flag it, so anything found by just
one is where the disagreement lives. Scan this list for both kinds of mistake: words that
are not personal information (add them to `SKIPLIST_TERMS`, section 6), and near-misses
that hint at a detail the other two also missed.


In [ ]:
agreement = (
    flagged.groupby(["doc_id", "text"])
    .agg(n_engines=("engine", "nunique"), max_score=("score", "max"),
         types=("entity_type", lambda values: sorted(set(values))))
    .reset_index()
    .sort_values(["n_engines", "max_score"])
)

agreement.head(30)


### 5.3 Search the output for identifiers you know are in your files

**The single most useful check in this notebook.** Take a few names, phone numbers or ID
numbers you know appear in your own documents — that is `MUST_NOT_APPEAR` in section 0 —
and confirm none of them survived. Replacement only covers what was detected, so a missed
detail is still sitting there in plain text.

Anything reported as `STILL PRESENT` has to be dealt with before the text goes anywhere:
add rules in section 7, or fix it by hand.


In [ ]:
for needle in MUST_NOT_APPEAR:
    hits = [doc_id for doc_id, text in anonymized.items() if needle in text]
    print(f"{needle!r}: {'STILL PRESENT in ' + str(hits) if hits else 'not found ✓'}")


### 5.4 The replacement log (optional)

`PII_Log.json` records where in the original document each replacement happened, which is
what you need if you want to compare the original and the cleaned version side by side.


In [ ]:
log_rows = [
    {"doc_id": doc_id, **entry}
    for doc_id, entries in pii_log.items()
    for entry in entries
]

pd.DataFrame(log_rows).head(20)


## 6. Stop it deleting clinical words

> **Optional**, but almost everyone needs it after a first look at section 5.1.

**TL;DR** — the detectors have never seen words like `Asperger` or `Ritalin`, so they
sometimes guess "that must be a person's name" and delete them. A *skiplist* is a list of
words that must never be removed. Put yours in `SKIPLIST_TERMS` (section 0) and run the
cell below; then re-run section 4 and check again.

The tool merges **every `.txt` file** in `data/external` into one skiplist, so you add
your own file and never touch the ones that ship with it. Matching ignores upper/lower
case but compares the **whole** flagged phrase, so write each term exactly as it appears
in the `text` column of section 5.1 — half a phrase will not match.


In [ ]:
skiplist_file = ANON_DIR / "data" / "external" / "project_skiplist.txt"
skiplist_file.parent.mkdir(parents=True, exist_ok=True)
skiplist_file.write_text("\n".join(SKIPLIST_TERMS) + "\n", encoding="utf-8")

print(f"{len(SKIPLIST_TERMS)} terms written to {skiplist_file}")
print("Re-run section 4, then check section 5 again.")


### Two blunter knobs (advanced)

Both live in `src/anonymize_pii/config.py` and need a text editor.

- **`timewords` and `generalwords`** drop any flagged phrase that *contains* one of those
  words — heavier-handed than the skiplist, useful for recurring patterns like date
  fragments or product names.
- **`Entities`** is the list of what is looked for at all. Removing a type raises
  precision: e.g. drop `DATE_TIME` if dates matter for your analysis, or the `US_*` types
  if your data is not American and they only produce noise. Adding a type only helps if
  the detectors actually support it.

After any change, re-run section 4 and re-check section 5. Two or three rounds is normal.


## 7. Catch names the detectors missed

> **Optional**, but recommended if your documents are clinical records, forms, or contain
> ALL-CAPS text. Skip it for ordinary prose in mixed case.

**TL;DR** — names are the hardest thing to get right, and the mistake is one-sided: a
missed name stays in the text, while a wrongly-flagged word only costs you one word of
content. This section adds a small set of rules that run *after* the detectors and close
the three gaps they most often leave. Everything here runs on public data plus a short
invented record — no sensitive material required.

**First, what does *not* go wrong.** It is often said that these detectors chop long
names in half, catching only the first given name. On well-formed mixed-case text that is
mostly **not** what happens:

```
"Se comunicó con María José García Pérez, madre del paciente."
    → PER: ['María José García Pérez']              ✓ all four words
"Refiere que Ana Paula Quispe Mamani la acompañó a la guardia."
    → PER: ['Ana Paula Quispe Mamani']              ✓
```

The three real failures are these.

**1. ALL-CAPS text flips the labels.** Administrative records are full of upper-case
fields, and the detectors fall apart on them:

```
"SE COMUNICO CON MARIA JOSE GARCIA PEREZ, MADRE DEL PACIENTE."
    → LOC: ['SE COMUNICO', 'CON MARIA JOSE GARCIA PEREZ']   ✗ the name is a PLACE
    → PER: ['MADRE DEL PACIENTE']                           ✗ and a non-name is the PERSON
```

The name is still removed here — the tool masks every type it tracks, so a name labelled
`LOCATION` gets masked anyway. What you lose is the ability to trust the labels, plus
anything the detectors drop completely.

**2. Form layouts make the name run into the next field.** In
`Paciente: <name>\nAcompañante: <name>`, what comes back is
`María José García Pérez\nAcompañante` — the *next* field's label is now part of the
"name".

**3. Titles get swallowed.** `La Lic. Ana Lucía Ramírez ...` comes back as
`Lic. Ana Lucía Ramírez`, title included.


### 7.1 Measuring the problem on public data (optional demo)

These demonstration cells need spaCy in *this* notebook's environment
(`pip install spacy` and `python -m spacy download en_core_web_lg`). They exist to show
you the size of the problem; in a real run you take the detections from `Iterator.json`
instead, as in 7.4. **Skip straight to 7.2 if you just want the rules.**

The sample is the set of Reddit posts that ships with `llm_tracker` (60 posts from
r/anxiety, r/depression and r/autism).


In [ ]:
reddit = pd.read_csv("../sample_data/reddit_autism_anxiety_depression.csv")

# Fallback if you are running outside a repo checkout (e.g. a bare Colab runtime):
# reddit = pd.read_csv(
#     "https://mair.sites.fas.harvard.edu/datasets/rmhd_27subreddits_1300posts_train.csv",
#     index_col=0,
# )

posts = reddit["post"].astype(str).tolist()
print(f"{len(posts)} posts, {sum(len(p) for p in posts):,} characters")


In [ ]:
import spacy

nlp = spacy.load("en_core_web_lg")
docs = list(nlp.pipe(posts))

person_spans = sorted({e.text for d in docs for e in d.ents if e.label_ == "PERSON"})
print(f"{len(person_spans)} distinct PERSON spans:")
print(person_spans)


Nine distinct results, of which four are wrong:

```
['Asperger', 'Brandon Sanderson', 'Nicholas Hoult', 'Sanderson', 'Sid',
 'Skins', 'Yada Yada', 'aspergers', 'dyspraxia']
```

`Asperger`, `aspergers`, `dyspraxia` and `Skins` are **not names** — they are diagnostic
vocabulary and a TV series. This is the commonest failure on clinical text, and exactly
what the skiplist in section 6 is for. Note also that `Brandon Sanderson` and `Sanderson`
are counted as two different people.

Now the ALL-CAPS experiment, on the five posts that contain at least one name:


In [ ]:
with_names = [p for p, d in zip(posts, docs)
              if any(e.label_ == "PERSON" for e in d.ents)]

def person_set(texts):
    return {e.text for d in nlp.pipe(texts) for e in d.ents if e.label_ == "PERSON"}

original = person_set(with_names)
uppercased = person_set([p.upper() for p in with_names])

print(f"{len(with_names)} posts contain a name")
print(f"original  : {len(original):2d} -> {sorted(original)}")
print(f"UPPERCASED: {len(uppercased):2d} -> {sorted(uppercased)}")


Nine results become seven, and not by simply losing two:

| | found |
| --- | --- |
| original | `Asperger`, `Brandon Sanderson`, `Nicholas Hoult`, `Sanderson`, `Sid`, `Skins`, `Yada Yada`, `aspergers`, `dyspraxia` |
| uppercased | `ASPERGER`, `BRANDON SANDERSON`, `DYSPRAXIA`, `I'VE`, `LIP`, `NICHOLAS HOULT`, `SANDERSON` |

Two real names (`Sid`, `Yada Yada`) are **lost**, and two new mistakes appear (`I'VE`,
`LIP`). Capital letters are a large part of how these detectors recognise a name, so
upper-casing moves both accuracy and coverage — which is also why **converting your text
to Title Case to "help" them is a bad trade**: you gain a few names and gain more noise.


### 7.2 The rules

**TL;DR** — in record-style text, names sit in predictable places, and the rules exploit
that. Three ingredients:

1. **Anchors** — a title (`Dr.`, `Lic.`, `Enf.`) or a field label (`Paciente:`,
   `Nombre y Apellido:`) is followed by a name. This still works in ALL-CAPS text, where
   the detectors do not.
2. **A guard list** — words a name may never absorb: articles, prepositions, clinical
   vocabulary, institution words, months and weekdays, and the title and label words
   themselves. Without that last group the rules eat the next field's label.
3. **Growing to a stable point** — each known name is extended over neighbouring
   capitalised words, again and again, until nothing changes. Repeating is what handles
   two given names *and* two surnames: `María` → `María José` → `María José García` →
   `María José García Pérez` takes several rounds. The length is capped (five words) so a
   runaway match cannot swallow a sentence.

The same code also **trims** results, which is what fixes the run-on into the next line
and the swallowed titles.

**This is the other list you may want to edit**: `stopwords` below. Add any ordinary word
that the rules wrongly glue onto a name in your own data.


In [ ]:
CAP = r"[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+"      # one capitalised, accent-aware word
MAX_NAME_TOKENS = 5                    # two given names + two surnames, plus slack

SPANISH = {
    "titles": ["Dr", "Dra", "Lic", "Licenciada", "Licenciado", "Enf", "Sr", "Sra",
               "Srta", "Prof", "Psic"],
    "labels": ["Nombre y Apellido", "Nombre", "Apellido", "Paciente", "Acompañante",
               "Responsable", "Madre", "Padre", "Tutor", "Derivado por"],
    "stopwords": {"El", "La", "Los", "Las", "Un", "Una", "Se", "Su", "Sus", "Del", "De",
                  "Y", "En", "Con", "Por", "Para", "No", "Si", "Hospital", "Centro",
                  "Salud", "Mental", "Guardia", "Servicio", "Femenino", "Masculino",
                  "Lunes", "Martes", "Enero", "Febrero"},
}

ENGLISH = {
    "titles": ["Dr", "Doctor", "Mr", "Mrs", "Ms", "Prof", "Nurse", "Therapist"],
    "labels": ["Name", "Patient", "Client", "Contact", "Referred by"],
    "stopwords": {"I", "The", "This", "That", "My", "But", "And", "So", "It", "He", "She",
                  "They", "We", "You", "A", "An", "If", "When", "What", "Just", "Then",
                  "Now", "There", "Here", "Also", "Autism", "Autistic", "Asperger",
                  "Aspergers", "Anxiety", "Depression", "ADHD", "OCD", "PTSD", "Monday",
                  "January", "University", "College", "School", "Please", "Help",
                  "Thanks", "Edit"},
}

RULES = SPANISH if LANGUAGE == "es" else ENGLISH      # which set the cells below use


def guard(cfg):
    """Words a name may never absorb, upper-cased so case does not matter.

    Titles and labels are guards too: without them the rules swallow the next
    field's label ("María José García Pérez Acompañante").
    """
    words = set(cfg["stopwords"]) | set(cfg["titles"])
    for label in cfg["labels"]:
        words |= set(label.split())
    return {w.upper() for w in words}


def _trim(tokens, blocked):
    """Keep the leading run of name-like words, stopping at the first guard word."""
    kept = []
    for token in tokens[:MAX_NAME_TOKENS]:
        if token.upper().strip(".,;:") in blocked:
            break
        kept.append(token)
    return kept


def anchors_from_rules(text, cfg):
    """Names sitting after a title or a field label, in mixed case *or* ALL CAPS."""
    titles = "|".join(re.escape(t) for t in cfg["titles"])
    labels = "|".join(re.escape(la) for la in cfg["labels"])
    patterns = [
        re.compile(rf"\b(?:{titles})\.?\s+({CAP}(?:\s+{CAP})*)"),        # Dra. Ana Lucía
        re.compile(rf"\b(?:{labels})\s*:\s*([^\n,;.]+)"),                 # Paciente: ...
        re.compile(                                                        # ALL-CAPS variant
            rf"\b(?:(?:{titles.upper()})\.?|(?:{labels.upper()})\s*:)\s+"
            rf"([A-ZÁÉÍÓÚÑ]{{2,}}(?:\s+[A-ZÁÉÍÓÚÑ]{{2,}})*)"
        ),
    ]
    blocked = guard(cfg)
    found = set()
    for pattern in patterns:
        for match in pattern.finditer(text):
            tokens = _trim(match.group(1).strip().split(), blocked)
            if tokens:
                found.add(" ".join(tokens))
    return found


def extend_spans(text, known, cfg, max_passes=5):
    """Trim the starting points, then grow them over neighbouring capitalised words."""
    blocked = guard(cfg)

    # Clean the starting points first: detector results run across line breaks (picking
    # up the next field's label) and, in ALL-CAPS text, carry on into ordinary words.
    spans = set()
    for seed in known:
        tokens = _trim(seed.split("\n")[0].strip().split(), blocked)
        if tokens:
            spans.add(" ".join(tokens))

    for _ in range(max_passes):
        grown = set()
        for name in spans:
            # In ALL-CAPS text the mixed-case pattern never matches: take the shape
            # of the words from the starting point itself.
            shape = r"[A-ZÁÉÍÓÚÑ]{2,}" if name.isupper() else CAP
            pattern = re.compile(
                rf"((?:{shape}[ \t]+)*){re.escape(name)}((?:[ \t]+{shape})*)"
            )
            for match in pattern.finditer(text):
                left = _trim(match.group(1).split()[::-1], blocked)[::-1]
                right = _trim(match.group(2).split(), blocked)
                candidate = " ".join([*left, name, *right]).strip()
                if candidate != name and len(candidate.split()) <= MAX_NAME_TOKENS:
                    grown.add(candidate)
        if not grown - spans:
            break
        spans |= grown

    # Keep only the longest version, so "Sanderson" gives way to "Brandon Sanderson".
    return {s for s in spans if not any(s != other and s in other for other in spans)}


def recover_names(text, seeds, cfg):
    """The full pass: detector results + rule anchors, trimmed and extended."""
    return extend_spans(text, set(seeds) | anchors_from_rules(text, cfg), cfg)


### 7.3 What the rules do and do not add

Run them over the Reddit posts, starting from what spaCy found:


In [ ]:
added = {}
for post, doc in zip(posts, docs):
    seeds = {e.text for e in doc.ents if e.label_ == "PERSON"}
    if not seeds:
        continue
    extra = recover_names(post, seeds, ENGLISH) - seeds
    if extra:
        added[post[:60]] = extra

print(added or "no spans added")


**Nothing is added — and that is the right answer.** These are conversational posts: no
titles, no field labels, and the names in them are already complete. A rule pass that
stays quiet here is a rule pass that will not flood your own records with invented names
either.

The rules earn their keep on record-style text. Here is the same pass over a short
invented record (made-up names, Spanish, the layout that breaks the detectors), in mixed
case and in ALL CAPS:


In [ ]:
DEMO = """Informe de guardia
Paciente: María José García Pérez
Acompañante: Juan Carlos Gómez Fernández
Evaluada por la Dra. Ana Lucía Ramírez en el Hospital San Roque.
La paciente refiere ansiedad. Se deriva a Salud Mental.
"""

nlp_es = spacy.load("es_core_news_lg")   # python -m spacy download es_core_news_lg

for label, text in (("mixed case", DEMO), ("ALL CAPS", DEMO.upper())):
    doc = nlp_es(text)
    seeds = {e.text for e in doc.ents if e.label_ == "PER"}
    print(f"--- {label} ---")
    print(f"  detector results : {sorted(seeds)}")
    print(f"  rule anchors     : {sorted(anchors_from_rules(text, SPANISH))}")
    print(f"  after the rules  : {sorted(recover_names(text, seeds, SPANISH))}")


Both versions end up with the same three names, and the artefacts are gone:

```
--- mixed case ---
  detector results : ['Juan Carlos Gómez Fernández\nEvaluada por la Dra. Ana Lucía Ramírez',
                      'María José García Pérez\nAcompañante']       ← ran across two lines
  after the rules  : ['Ana Lucía Ramírez', 'Juan Carlos Gómez Fernández',
                      'María José García Pérez']                    ✓

--- ALL CAPS ---
  detector results : ['ANA LUCÍA RAMÍREZ EN EL', 'JUAN CARLOS GÓMEZ FERNÁNDEZ',
                      'MARÍA JOSÉ GARCÍA PÉREZ']                    ← ran into ordinary words
  after the rules  : ['ANA LUCÍA RAMÍREZ', 'JUAN CARLOS GÓMEZ FERNÁNDEZ',
                      'MARÍA JOSÉ GARCÍA PÉREZ']                    ✓
```

Look at `ANA LUCÍA RAMÍREZ`: the detector ran the name into `EN EL`, the guard list
stopped it at `EN`, and the `Dra.` anchor confirmed where the name began.
`HOSPITAL SAN ROQUE` was never absorbed because `Hospital` is a guard word — an
institution, not a person.


### 7.4 Applying the recovered names to your own output

**TL;DR** — the tool has no way to accept an extra list of words, so the recovered names
are applied as a second replacement pass over its output. Two things matter: start from
the **original** text (that is where the names still are), and replace the **longest name
first**, so `María José García Pérez` is masked before a bare `María` can cut it in half.

In a real run the starting points come from `Iterator.json` — the pipeline already did
that detection work, so there is no need to run spaCy again.


In [ ]:
def person_seeds_from_iterator(iterator, doc_id, engines=("spacy", "stanza", "GLiNER")):
    """The spans the pipeline labelled PERSON for one document."""
    return {
        text
        for engine in engines
        for text, (entity_type, _score) in iterator.get(doc_id, {}).get(engine, {}).items()
        if entity_type == "PERSON"
    }


def _fragments(name, blocked):
    """Pieces of a name, longest first.

    The whole name is often no longer in the cleaned text: if the pipeline caught only
    the first given name, "María José García Pérez" survives as
    "<PERSON> José García Pérez", so the leftover has to be matched on its own.
    """
    tokens = name.split()
    spans = [
        tokens[start:end]
        for size in range(len(tokens), 0, -1)
        for start in range(0, len(tokens) - size + 1)
        for end in (start + size,)
    ]
    return [
        " ".join(span)
        for span in spans
        if not (len(span) == 1 and (len(span[0]) < 3 or span[0].upper() in blocked))
    ]


def apply_recovered_names(original, anonymized, seeds, cfg, replacement="<PERSON>"):
    """Mask the names the rules recovered, including ones only partly masked."""
    recovered = recover_names(original, seeds, cfg)
    blocked = guard(cfg)
    masked = set()

    for name in sorted(recovered, key=len, reverse=True):
        fragments = _fragments(name, blocked)
        for _ in range(len(fragments)):          # bounded: each round removes text
            hit = next(
                (f for f in fragments if f in anonymized.replace(replacement, "")),
                None,
            )
            if hit is None:
                break
            anonymized = anonymized.replace(hit, replacement)
            masked.add(hit)

    return anonymized, {"recovered": recovered, "masked": masked}


# reports = what you fed the pipeline; anonymized = what it gave back
patched, report_log = {}, {}
for doc_id, anon_text in anonymized.items():
    seeds = person_seeds_from_iterator(iterator, doc_id)
    patched[doc_id], report_log[doc_id] = apply_recovered_names(
        reports[doc_id], anon_text, seeds, RULES
    )

still_leaking = {k: v["masked"] for k, v in report_log.items() if v["masked"]}
print(f"{len(still_leaking)} documents had names the pipeline left in plain text")
for doc_id, spans in list(still_leaking.items())[:10]:
    print(f"  {doc_id}: {sorted(spans)}")


**Read `report_log` before trusting `patched`.** Each entry says what the rules decided
was a name (`recovered`) and what actually still needed masking (`masked`) — a rule pass
is only as good as its guard list. Add every wrongly-caught word to `stopwords` in 7.2
and run it again: the same review loop as section 6, and worth two or three rounds on a
new set of documents.

From here on, `patched` is the text you send onward. Keep `report_log` as a record of what
this second pass caught.

> If you ran this section, use `patched` instead of `anonymized` in section 8.


## 8. Send the clean text to `llm_tracker`

> Required only if `llm_tracker` is where you are heading. If you only needed the
> anonymised files, stop after section 9 — they are in `data/exports`.

**TL;DR** — turn the cleaned documents back into a CSV and hand it to the analyser.
`analyze_csv()` builds each document identifier by joining two columns with `_`, so
keeping your identifiers in one of them preserves the trail back to the original records.

The cell below picks up the output of section 7 if you ran it, and the plain pipeline
output if you did not.


In [ ]:
final_documents = patched if "patched" in globals() else anonymized

anon_df = pd.DataFrame(
    {
        "source": "anonymized",                  # first half of the llm_tracker ID
        "doc_id": list(final_documents),         # second half — your own identifiers
        "text": list(final_documents.values()),
    }
)

anon_csv = Path("anonymized_documents.csv")
anon_df.to_csv(anon_csv, index=False)
print(f"{len(anon_df)} documents -> {anon_csv}")
anon_df.head()


In [ ]:
from llm_tracker import AnalyzerConfig, LLMTrackerAnalyzer

config = AnalyzerConfig(
    api_key=OPENROUTER_API_KEY,
    model_name=LLM_MODEL,
)
analyzer = LLMTrackerAnalyzer(config=config)

results_llm, metadata_llm, errors_llm = analyzer.analyze_csv(
    csv_path=str(anon_csv),
    codebook_path=CODEBOOK_PATH,
    text_column="text",
    subreddit_column="source",     # document IDs come out as "anonymized_<doc_id>"
    author_column="doc_id",
    output_dir="LLM_coding",
)


From here on, continue with [`tutorial.ipynb`](../tutorial.ipynb) — comparison against
human coding, summary tables, and metrics.

`llm_tracker` can also read a folder of `.txt` files directly, with
`analyzer.analyze_directory(input_dir=..., codebook_path=...)`. Use it on a folder of
**already anonymised** files; never on the originals.


## 9. Checklist before anything leaves your computer

- [ ] Every document went through the anonymiser (no half-finished batches).
- [ ] You looked at the low-confidence detections and the ones only one detector found
      (sections 5.1 and 5.2).
- [ ] The identifiers in `MUST_NOT_APPEAR` no longer appear in the output (section 5.3).
- [ ] If your text is record-style or upper-case, you ran the name rules and reviewed
      what they recovered (section 7).
- [ ] Columns with fixed values were de-identified with your own rules, not with the
      detectors.
- [ ] Free-text fields you never meant to send are out of the CSV entirely.
- [ ] A person read at least a sample of the cleaned documents from beginning to end.
- [ ] The original, un-anonymised files are not in a folder that is synced, shared, or
      under version control.


## Appendix A: how the pipeline works

Worth reading when a result surprises you.

**1. Three detectors read every document independently.**

| detector | model | what it adds |
| --- | --- | --- |
| `spacy` | `en_core_web_lg` | fast, statistical |
| `stanza` | `en` | built differently, so it catches different things |
| `GLiNER` | `nvidia/gliner-pii` | works from a description of what to look for; reads the text in 512-character chunks |

`spacy` and `stanza` also load a set of fixed rules for emails, phone numbers, credit
cards and US identity documents, so the run mixes "the model thinks this is a name" with
"this matches the pattern of an email address". What is searched for at all is the
`Entities` list in `config.py`.

**2. The findings are pooled, and the most confident label wins.** When several detectors
flag the same words, the label from the highest-scoring one is kept. Detection is a
**union** — one detector is enough for something to be flagged.

**3. A filter drops known-safe words.** Anything on the skiplist, or containing a time
word (`day`, `morning`, `age`…) or a general word (`DSM-5`, `zoom`…), is put back. This is
the part you control in section 6.

**4. Replacement works from a list of strings.** The surviving strings are collected and
the document is searched again, so **every** occurrence of a flagged string is replaced,
not just the one that was spotted.

The consequence that matters, and the reason section 5 exists: **anything the detectors
missed stays in the text word for word.**


## Appendix B: anonymising only part of a document

> Advanced and optional. Needs `headhunter` installed (section 1).

With `--parse`, the input is first tidied up by
[`headhunter`](https://github.com/childmindresearch/headhunter). This is what you want
when your documents are long, inconsistently formatted reports and you only need **certain
sections** anonymised, or when your input is a table. Configure it through
`headhunter_config` in `config.py`; the intermediate result is written to
`data/parsed/Parsed_Reports.json`.

The mode is worked out from the input:

| `input_path` | `content_columns` | mode |
| --- | --- | --- |
| `.json` (`{id: text}`) | ignored | JSON |
| `.csv` / `.parquet` | exactly one | single column |
| `.csv` / `.parquet` | several | multi-column |

```python
headhunter_config = {
    'input_path': str(report_in / 'my_reports.csv'),
    'content_columns': ['report'],
    'id_column': 'report_id',
    'parser_config': {'heading_max_words': 10},
    'expected_headings': None,
    'match_threshold': 80,
    'headings_to_anonymize': ['clinical summary', 'treatment plan'],
    'separate_headings_into_reports': False,
}
```

Before you spend time on it:

- Leaving `headings_to_anonymize` empty means the whole document is anonymised.
- Filling it in keeps **only** the matching sections, so **everything under the other
  headings disappears from the output**. It is a filter, not a mask.
- `separate_headings_into_reports=True` produces one entry per matched section, named
  `{id}/{heading}`, instead of merging them into one document.
- `parser_config`, `expected_headings` and `match_threshold` apply to JSON and
  single-column mode only; they are ignored with several columns.
